In [ ]:
import os
import cv2
import numpy as np
import joblib
from deepface import DeepFace
from time import time

# --- 💡 LÍMITE DE IMÁGENES 💡 ---
# Pon 500 para una prueba decente, o 200 para una prueba súper rápida
MAX_IMAGES_PER_CLASS = 500 

print(f"Script 1: Extracción de Embeddings (MODO RÁPIDO: max {MAX_IMAGES_PER_CLASS} por clase)")

# --- Función LoadDataset (Adaptada de tu Deepface_kfold.ipynb) ---
def LoadDataset(folder, ext, max_per_class):
    nclasses = 0
    nperclass = []
    classlabels = []
    X = []
    Y = []

    print(f"Cargando dataset desde: {folder}")
    
    # Obtener lista de clases (directorios)
    class_list = [d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d))]
    
    for class_name in class_list:
        class_folder = os.path.join(folder, class_name)
            
        nclasses += 1
        nsamples = 0
        print(f"\nCargando clase: {class_name} ({nclasses}/{len(class_list)})")

        for file_name in os.listdir(class_folder):
            
            # --- 💡 LÓGICA DEL LÍMITE 💡 ---
            if nsamples >= max_per_class:
                print(f"   ... Límite alcanzado ({max_per_class} imágenes)")
                break # Rompe el bucle de esta clase y pasa a la siguiente

            if file_name.endswith(ext):
                image_path = os.path.join(class_folder, file_name)
                try:
                    image = cv2.imread(image_path)
                    if image is None:
                        continue
                        
                    img1 = cv2.resize(image, dim, interpolation=cv2.INTER_AREA)

                    embedding_objs = DeepFace.represent(
                        img_path=img1,
                        model_name=model_name,
                        enforce_detection=False
                    )
                    img_embedding = embedding_objs[0]["embedding"]
                    
                    X.append(img_embedding)
                    Y.append(nclasses - 1)
                    nsamples += 1
                    
                    if nsamples % 100 == 0:
                        print(f"\r   ... procesadas {nsamples} imágenes", end="")
                
                except Exception as e:
                    # Ignora errores de 'represent' (ej. cara no encontrada)
                    pass

        print(f"\r   -> Clase '{class_name}' completada. Total: {nsamples} imágenes.")
        nperclass.append(nsamples)
        classlabels.append(class_name)

    X = np.array(X, dtype='float32')
    Y = np.array(Y, dtype='float64')

    if X.size == 0:
        return X, Y, 0, 0, 0, [], [], []

    n_samples, n_features = X.shape
    class_names = np.array(classlabels)
    n_classes = class_names.shape[0]

    return X, Y, n_samples, n_features, n_classes, classlabels, nperclass, class_names

# --- 1. Configuración del Modelo DeepFace ---
model_name = "Facenet"
print(f"Construyendo modelo: {model_name}")
model = DeepFace.build_model(model_name)
dim = (model.input_shape[1], model.input_shape[0]) 
print(f"Dimensiones de entrada: {dim}")

# --- 2. Carga del Dataset ---
folder = "/content/train" 

# ¡Ahora pasamos el límite como argumento!
X, Y, n_samples, n_features, n_classes, classlabels, nperclass, class_names = LoadDataset(folder, '.png', MAX_IMAGES_PER_CLASS)

print("\n--- Información del Dataset ---")
print(f"# Muestras: {n_samples}")
print(f"# Características (Embeddings): {n_features}")
print(f"# Clases: {n_classes}")

# --- 3. GUARDAR LOS DATOS EXTRAÍDOS ---
if n_samples > 0:
    print("\nGuardando datos extraídos en archivos .pkl...")
    
    joblib.dump(X, 'embeddings_X.pkl')
    joblib.dump(Y, 'labels_Y.pkl')
    joblib.dump(class_names, 'emotion_class_names.pkl')
    
    print("¡Éxito! Archivos 'embeddings_X.pkl', 'labels_Y.pkl' y 'emotion_class_names.pkl' guardados.")
    print("Ahora puedes ejecutar '2_entrenar_svm.py'.")
else:
    print("Error: No se cargaron muestras. Verifica la ruta de tu dataset ('folder') y la extensión ('.png').")

In [ ]:
import numpy as np
import joblib
from time import time
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

print("Script 2: Entrenamiento del Modelo SVM (Rápido)")

# --- 1. Cargar datos pre-extraídos ---
try:
    print("Cargando 'embeddings_X.pkl' y 'labels_Y.pkl'...")
    X = joblib.load('embeddings_X.pkl')
    Y = joblib.load('labels_Y.pkl')
    print(f"Datos cargados: {X.shape[0]} muestras, {X.shape[1]} características.")
except FileNotFoundError:
    print("Error: No se encontraron los archivos .pkl.")
    print("Por favor, ejecuta '1_extraer_embeddings.py' primero.")
    exit()

# --- 2. Entrenamiento del Modelo SVM ---
if X.shape[0] > 0:
    print("\nEntrenando Scaler (MinMaxScaler)...")
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    print("Iniciando GridSearchCV para SVM...")
    t0 = time()
    parameters = {'C': [1e3, 5e3, 1e4], 
                  'gamma': [0.0001, 0.001, 0.01]} 
    
    clf = GridSearchCV(
        SVC(kernel='rbf', class_weight='balanced', probability=True), 
        parameters, 
        cv=3,
        n_jobs=-1, # Usar todos los cores (ahora irá rápido)
        verbose=3  # Imprime el progreso
    )
    clf.fit(X_scaled, Y)
    
    print(f"GridSearchCV terminado en {time() - t0:.3f}s")
    print("Mejor estimador encontrado:")
    print(clf.best_estimator_)
    
    # 2.3 Guardar los 2 modelos finales para el prototipo
    final_model = clf.best_estimator_
    print("\nGuardando modelos finales...")
    
    joblib.dump(final_model, 'emotion_svm_model.pkl')
    joblib.dump(scaler, 'emotion_scaler.pkl')
    
    print("¡Éxito! Archivos 'emotion_svm_model.pkl' y 'emotion_scaler.pkl' guardados.")
    print("¡Todo listo para ejecutar el prototipo!")
else:
    print("Error: Los datos cargados están vacíos.")

TODO

In [ ]:
import os
import cv2
import numpy as np
import joblib
from deepface import DeepFace
from time import time
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

print("Script de Entrenamiento de Emociones")

# --- Función LoadDataset (Adaptada de tu Deepface_kfold.ipynb) ---
# Esta función necesita que 'model' y 'dim' estén definidos globalmente
def LoadDataset(folder, ext):
    nclasses = 0
    nperclass = []
    classlabels = []
    X = []
    Y = []

    print(f"Cargando dataset desde: {folder}")
    
    for class_name in os.listdir(folder):
        class_folder = os.path.join(folder, class_name)
        if not os.path.isdir(class_folder):
            continue
            
        nclasses += 1
        nsamples = 0
        print(f"Cargando clase: {class_name} ({nclasses})")

        for file_name in os.listdir(class_folder):
            if file_name.endswith(ext):
                image_path = os.path.join(class_folder, file_name)
                try:
                    image = cv2.imread(image_path)
                    if image is None:
                        print(f"Advertencia: No se pudo leer {image_path}")
                        continue
                        
                    # Redimensionar la imagen a la entrada del modelo
                    img1 = cv2.resize(image, dim, interpolation=cv2.INTER_AREA)

                    # Obtener embedding (usando el model_name global)
                    embedding_objs = DeepFace.represent(
                        img_path=img1,
                        model_name=model_name,
                        enforce_detection=False
                    )
                    img_embedding = embedding_objs[0]["embedding"]
                    
                    X.append(img_embedding)
                    Y.append(nclasses - 1)
                    nsamples += 1
                
                except Exception as e:
                    print(f"Error procesando {image_path}: {e}")

        nperclass.append(nsamples)
        classlabels.append(class_name)

    X = np.array(X, dtype='float32')
    Y = np.array(Y, dtype='float64')

    n_samples, n_features = X.shape
    class_names = np.array(classlabels)
    n_classes = class_names.shape[0]

    return X, Y, n_samples, n_features, n_classes, classlabels, nperclass, class_names

# --- 1. Configuración del Modelo DeepFace ---
model_name = "Facenet"
print(f"Construyendo modelo: {model_name}")
model = DeepFace.build_model(model_name)
# 'dim' debe ser (ancho, alto), pero input_shape es (alto, ancho, canales)
dim = (model.input_shape[1], model.input_shape[0]) 
print(f"Dimensiones de entrada: {dim}")

# --- 2. Carga del Dataset ---
# ⬇️⬇️⬇️ ¡MODIFICA ESTA LÍNEA! ⬇️⬇️⬇️
folder = "C:/ruta/a/tu/dataset_emociones/train"  # Ejemplo: "D:/datasets/FER2013/train"
# ⬆️⬆️⬆️ ¡MODIFICA ESTA LÍNEA! ⬆️⬆️⬆️

X, Y, n_samples, n_features, n_classes, classlabels, nperclass, class_names = LoadDataset(folder, '.jpg') # Asumimos .jpg, cambia si es .png

print("\n--- Información del Dataset ---")
print(f"# Muestras: {n_samples}")
print(f"# Características (Embeddings): {n_features}")
print(f"# Clases: {n_classes}")
print(f"Clases: {classlabels}")
print(f"Muestras por clase: {nperclass}")
print("---------------------------------")

# --- 3. Entrenamiento del Modelo Final ---
if n_samples > 0:
    # 3.1 Normalizar los datos
    print("\nEntrenando Scaler (MinMaxScaler)...")
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    # 3.2 Buscar los mejores parámetros para el SVM
    print("Iniciando GridSearchCV para SVM...")
    t0 = time()
    # Reducimos los parámetros para que sea más rápido
    parameters = {'C': [1e3, 5e3, 1e4], 
                  'gamma': [0.0001, 0.001, 0.01]} 
    
    # Usamos cv=3 (3-fold cross-validation) para más velocidad
    clf = GridSearchCV(
        SVC(kernel='rbf', class_weight='balanced', probability=True), 
        parameters, 
        cv=3,
        n_jobs=-1 # Usar todos los cores
    )
    clf.fit(X_scaled, Y)
    
    print(f"GridSearchCV terminado en {time() - t0:.3f}s")
    print("Mejor estimador encontrado:")
    print(clf.best_estimator_)
    
    # 3.3 Guardar los 3 archivos .pkl
    final_model = clf.best_estimator_
    print("\nGuardando modelos...")
    
    joblib.dump(final_model, 'emotion_svm_model.pkl')
    joblib.dump(scaler, 'emotion_scaler.pkl')
    joblib.dump(class_names, 'emotion_class_names.pkl')
    
    print("¡Éxito! Archivos 'emotion_svm_model.pkl', 'emotion_scaler.pkl' y 'emotion_class_names.pkl' guardados.")
else:
    print("Error: No se cargaron muestras. Verifica la ruta de tu dataset.")

In [ ]:
import os
import cv2
import numpy as np
import joblib
from deepface import DeepFace
from time import time
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

print("Script de Entrenamiento de Emociones")

# --- Función LoadDataset (Adaptada de tu Deepface_kfold.ipynb) ---
# Esta función necesita que 'model' y 'dim' estén definidos globalmente
def LoadDataset(folder, ext):
    nclasses = 0
    nperclass = []
    classlabels = []
    X = []
    Y = []

    print(f"Cargando dataset desde: {folder}")
    
    for class_name in os.listdir(folder):
        class_folder = os.path.join(folder, class_name)
        if not os.path.isdir(class_folder):
            continue
            
        nclasses += 1
        nsamples = 0
        print(f"Cargando clase: {class_name} ({nclasses})")

        for file_name in os.listdir(class_folder):
            if file_name.endswith(ext):
                image_path = os.path.join(class_folder, file_name)
                try:
                    image = cv2.imread(image_path)
                    if image is None:
                        print(f"Advertencia: No se pudo leer {image_path}")
                        continue
                        
                    # Redimensionar la imagen a la entrada del modelo
                    img1 = cv2.resize(image, dim, interpolation=cv2.INTER_AREA)

                    # Obtener embedding (usando el model_name global)
                    embedding_objs = DeepFace.represent(
                        img_path=img1,
                        model_name=model_name,
                        enforce_detection=False
                    )
                    img_embedding = embedding_objs[0]["embedding"]
                    
                    X.append(img_embedding)
                    Y.append(nclasses - 1)
                    nsamples += 1
                
                except Exception as e:
                    print(f"Error procesando {image_path}: {e}")

        nperclass.append(nsamples)
        classlabels.append(class_name)

    X = np.array(X, dtype='float32')
    Y = np.array(Y, dtype='float64')

    n_samples, n_features = X.shape
    class_names = np.array(classlabels)
    n_classes = class_names.shape[0]

    return X, Y, n_samples, n_features, n_classes, classlabels, nperclass, class_names

# --- 1. Configuración del Modelo DeepFace ---
model_name = "Facenet"
print(f"Construyendo modelo: {model_name}")
model = DeepFace.build_model(model_name)
# 'dim' debe ser (ancho, alto), pero input_shape es (alto, ancho, canales)
dim = (model.input_shape[1], model.input_shape[0]) 
print(f"Dimensiones de entrada: {dim}")

# --- 2. Carga del Dataset ---
# ⬇️⬇️⬇️ ¡MODIFICA ESTA LÍNEA! ⬇️⬇️⬇️
folder = "C:/ruta/a/tu/dataset_emociones/train"  # Ejemplo: "D:/datasets/FER2013/train"
# ⬆️⬆️⬆️ ¡MODIFICA ESTA LÍNEA! ⬆️⬆️⬆️

X, Y, n_samples, n_features, n_classes, classlabels, nperclass, class_names = LoadDataset(folder, '.jpg') # Asumimos .jpg, cambia si es .png

print("\n--- Información del Dataset ---")
print(f"# Muestras: {n_samples}")
print(f"# Características (Embeddings): {n_features}")
print(f"# Clases: {n_classes}")
print(f"Clases: {classlabels}")
print(f"Muestras por clase: {nperclass}")
print("---------------------------------")

# --- 3. Entrenamiento del Modelo Final ---
if n_samples > 0:
    # 3.1 Normalizar los datos
    print("\nEntrenando Scaler (MinMaxScaler)...")
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    # 3.2 Buscar los mejores parámetros para el SVM
    print("Iniciando GridSearchCV para SVM...")
    t0 = time()
    # Reducimos los parámetros para que sea más rápido
    parameters = {'C': [1e3, 5e3, 1e4], 
                  'gamma': [0.0001, 0.001, 0.01]} 
    
    # Usamos cv=3 (3-fold cross-validation) para más velocidad
    clf = GridSearchCV(
        SVC(kernel='rbf', class_weight='balanced', probability=True), 
        parameters, 
        cv=3,
        n_jobs=-1 # Usar todos los cores
    )
    clf.fit(X_scaled, Y)
    
    print(f"GridSearchCV terminado en {time() - t0:.3f}s")
    print("Mejor estimador encontrado:")
    print(clf.best_estimator_)
    
    # 3.3 Guardar los 3 archivos .pkl
    final_model = clf.best_estimator_
    print("\nGuardando modelos...")
    
    joblib.dump(final_model, 'emotion_svm_model.pkl')
    joblib.dump(scaler, 'emotion_scaler.pkl')
    joblib.dump(class_names, 'emotion_class_names.pkl')
    
    print("¡Éxito! Archivos 'emotion_svm_model.pkl', 'emotion_scaler.pkl' y 'emotion_class_names.pkl' guardados.")
else:
    print("Error: No se cargaron muestras. Verifica la ruta de tu dataset.")